# PIKAN prediction for the uniform infinite-domain problem

In [1]:
from pathlib import Path
import sys
from importlib import reload

import matplotlib.pyplot as plt
import numpy as np
import torch

# Find the repository utilities directory from either the notebook folder or workspace root.
notebook_dir = Path.cwd().resolve()
repo_root = next(
    (path for path in [notebook_dir, *notebook_dir.parents] if (path / "utils").is_dir()),
    notebook_dir,
)
utilities_dir = repo_root / "utils"
if str(utilities_dir) not in sys.path:
    sys.path.insert(0, str(utilities_dir))

import infinite
import pinns_infinite
reload(infinite)
reload(pinns_infinite)

from infinite import analytical_solution_inf, coefficient_inf, evaluate_model_inf
from pinns_infinite import build_models_KAN, set_seed, train_dual_network

set_seed(42)
torch.set_default_dtype(torch.float32)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

/home/orincon/miniconda3/envs/PIKAN-unbounded-domains-env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda


## Uniform-sampling PIKAN configuration

These values match the stored uniform-infinite KAN model: three hidden layers, 25 units per layer, grid size 5, spline order 3, and uniform sampling.

In [6]:
config = {
    # Choose "train" to fit a new model or "load" to use a stored checkpoint.
    "mode": "train",
    "checkpoint_name": "pikan_infinite_uniform_weights.pt",
    "hidden_layers": 3,
    "hidden_units": 25,
    "grid_size": 5,
    "spline_order": 3,
    "adam_lr": 1e-3,
    "adam_iters": 2000,
    "lbfgs_iters": 2000,
    "sampling": "uniform",
    "sigma": 5.5,
    "exp_scale": 4.0,
    "n_obs_u": 100,
    "n_obs_k": 100,
    "n_pde": 1000,
    "seed": 2,
    "pde_alpha": 0.5,
    "pde_beta": 5.0,
    "epsilon": 1.0,
}

for name, value in config.items():
    print(f"{name}: {value}")

mode: train
checkpoint_name: pikan_infinite_uniform_weights.pt
hidden_layers: 3
hidden_units: 25
grid_size: 5
spline_order: 3
adam_lr: 0.001
adam_iters: 2000
lbfgs_iters: 2000
sampling: uniform
sigma: 5.5
exp_scale: 4.0
n_obs_u: 100
n_obs_k: 100
n_pde: 1000
seed: 2
pde_alpha: 0.5
pde_beta: 5.0
epsilon: 1.0


## Build the KAN models

The two KANs learn the solution $u(x,y)$ and variable coefficient $k(x,y)$ jointly through the physics-informed loss.

In [7]:
model_u, model_k = build_models_KAN(
    device=device,
    hidden_layers=config["hidden_layers"],
    hidden_units=config["hidden_units"],
    grid_size=config["grid_size"],
    spline_order=config["spline_order"],
)

print(model_u)
print(model_k)

KAN(
  (layers): ModuleList(
    (0-3): 4 x KANLinear(
      (base_activation): SiLU()
    )
  )
)
KAN(
  (layers): ModuleList(
    (0-3): 4 x KANLinear(
      (base_activation): SiLU()
    )
  )
)


## Load the uniform-sampling PIKAN

This cell loads the stored uniform-infinite model; no retraining is performed.

In [8]:
results_dir = repo_root / "main" / "03_individual_prediction" / "results"
weights_path = results_dir / config["checkpoint_name"]
legacy_weights_dir = repo_root / "main" / "04_sampling_approximation" / "results" / "2026-08-31_17-21-33_uniform_adaptive_sched_no_reg"
available_checkpoints = sorted(results_dir.glob("*.pt"))
print("Available combined checkpoints:")
for checkpoint in available_checkpoints:
    print(f"  - {checkpoint.name}")
print(f"Legacy separate weights: {legacy_weights_dir}")

mode = config["mode"].lower()
if mode not in {"train", "load"}:
    raise ValueError('config["mode"] must be either "train" or "load"')

if mode == "load":
    if weights_path.exists():
        checkpoint = torch.load(weights_path, map_location=device)
        model_u.load_state_dict(checkpoint["model_u"])
        model_k.load_state_dict(checkpoint["model_k"])
        config.update(checkpoint.get("config", {}))
        metrics = checkpoint.get("metrics", {})
        print(f"Loaded stored model: {weights_path}")
    else:
        model_u_path = legacy_weights_dir / "model_u.pt"
        model_k_path = legacy_weights_dir / "model_k.pt"
        if not model_u_path.exists() or not model_k_path.exists():
            raise FileNotFoundError(
                f"Neither combined checkpoint nor legacy weights were found. "
                f"Checked {weights_path} and {legacy_weights_dir}"
            )
        model_u.load_state_dict(torch.load(model_u_path, map_location=device))
        model_k.load_state_dict(torch.load(model_k_path, map_location=device))
        print(f"Loaded legacy uniform-infinite weights: {legacy_weights_dir}")
else:
    history = train_dual_network(
        model_u,
        model_k,
        adam_lr=config["adam_lr"],
        adam_iters=config["adam_iters"],
        lbfgs_iters=config["lbfgs_iters"],
        verbose=True,
        print_every=100,
        save_every=100,
        lambda_pde_scheduler=True,
        adaptive_weights=True,
        alpha=7,
        update_every=100,
        regularization=False,
        sampling=config["sampling"],
        sigma=config["sigma"],
        exp_scale=config["exp_scale"],
        n_obs_u=config["n_obs_u"],
        n_obs_k=config["n_obs_k"],
        n_pde=config["n_pde"],
        seed=config["seed"],
        save_results=True,
        base_dir=str(results_dir),
        run_name="pikan_infinite_uniform",
        pde_alpha=config["pde_alpha"],
        pde_beta=config["pde_beta"],
        epsilon=config["epsilon"],
        device=device,
    )

model_u.eval()
model_k.eval()
print(model_u)
print(model_k)

Available combined checkpoints:
  - pikan_infinite_gaussian_weights.pt
  - pikan_infinite_tuned_weights.pt
  - pikan_semi_infinite_tuned_weights.pt
  - pikan_semi_infinite_uniform_weights.pt
Legacy separate weights: /home/orincon/unbounded-domains/main/04_sampling_approximation/results/2026-08-31_17-21-33_uniform_adaptive_sched_no_reg

Training with Adam
Adam     0 | Total=9.446e+00 | ObsU=7.671e-03 | ObsK=4.880e+00 | PDE=4.558e+01 | Ratio=636.14 | ErrU=1.038e+00 | ErrK=9.820e-01
V      = [0.008 4.88 ]
R      = [0. 1.]
ratio  = 636.14
lambda_u = 1.000
lambda_k = 453.295
lambda_pde = 0.100
Adam   100 | Total=4.648e+00 | ObsU=1.208e-02 | ObsK=1.150e-01 | PDE=4.521e+01 | Ratio=636.14 | ErrU=1.378e+00 | ErrK=1.504e-01
V      = [0.01  2.497]
R      = [0. 1.]
ratio  = 252.85
lambda_u = 1.000
lambda_k = 184.997
lambda_pde = 0.100
Adam   200 | Total=1.688e+00 | ObsU=4.698e-02 | ObsK=1.031e-03 | PDE=1.174e+01 | Ratio=252.85 | ErrU=2.682e+00 | ErrK=1.420e-02
V      = [0.022 1.665]
R      = [0. 1

## Evaluate against the analytical solution

Compute global, in-domain, and out-of-domain mean absolute errors for both learned fields.

In [10]:
evaluation = evaluate_model_inf(
    model_u=model_u,
    model_k=model_k,
    analytical_solution=analytical_solution_inf,
    coefficient=coefficient_inf,
    sampling=config["sampling"],
    train_xmin=-5.0,
    train_xmax=5.0,
    train_ymin=-5.0,
    train_ymax=5.0,
    eval_xmin=-10.0,
    eval_xmax=10.0,
    eval_ymin=-10.0,
    eval_ymax=10.0,
    n_grid=400,
    alpha=config["pde_alpha"],
    beta=config["pde_beta"],
    epsilon=config["epsilon"],
    device=device,
    verbose=True,
)

metric_names = [
    "err_u_global", "err_k_global",
    "err_u_inside", "err_k_inside",
    "err_u_outside", "err_k_outside",
]
metrics = {name: float(evaluation[name]) for name in metric_names}
metrics


Spatial generalization (MAE)
Training domain : [-5.0, 5.0] × [-5.0, 5.0]
Evaluation domain : [-10.0, 10.0] × [-10.0, 10.0]

Global MAE
u : 6.563e-02
k : 9.516e-02

Inside training domain
u : 4.234e-03
k : 9.742e-04

Outside training domain
u : 8.610e-02
k : 1.266e-01


{'err_u_global': 0.06563299582008132,
 'err_k_global': 0.09516133219007045,
 'err_u_inside': 0.004234044849704136,
 'err_k_inside': 0.0009741648171703514,
 'err_u_outside': 0.08609931281020705,
 'err_k_outside': 0.1265570546477038}